# Quesion 1 Part 3
Loaded the model which was fine tuned in previous question

In [157]:
!pip install speechbrain
!pip install museval
!pip install pesq
!pip install mir_eval

### Loading models

In [158]:
import os
import torch
import torchaudio
import librosa
import numpy as np
import pandas as pd
from speechbrain.inference.separation import SepformerSeparation as separator
from pesq import pesq
from tqdm import tqdm
import torch
from transformers import WavLMModel
import torch.nn as nn
from peft import LoraConfig, get_peft_model

# Check for GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load the WavLM model and processor for speaker identification
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Load Pretrained WavLM Model
model_name = "microsoft/wavlm-base-plus"
model_pretrained = WavLMModel.from_pretrained(model_name).to(device)



Using device: cuda


In [159]:
# Finetuned Model 
import torch
from peft import PeftModel, get_peft_model
from transformers import WavLMModel
import torch.nn as nn

# Apply LoRA configuration
config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "intermediate_dense", "output_dense"],
)
wavlm = get_peft_model(model_pretrained, config).to(device)  # Ensure LoRA is on device

# Define SpeakerClassifier (modified for embeddings)
class SpeakerClassifier(nn.Module):
    def __init__(self, base_model, embedding_dim=768, num_classes=100):
        super().__init__()
        self.base_model = base_model.to(device)  # Already float32
        self.fc = nn.Linear(embedding_dim, num_classes).to(device)
        
    def forward(self, input_values, attention_mask=None):
        # Convert input to float32 if needed
        if input_values.dtype != torch.float32:
            input_values = input_values.float()  # Force float32
        
        outputs = self.base_model(input_values, attention_mask=attention_mask)
        embeddings = outputs[0].mean(dim=1)
        return embeddings.float()  # Ensure output is float32

# Initialize and load pretrained weights
model_finetuned = SpeakerClassifier(wavlm)
model_finetuned.load_state_dict(torch.load("/kaggle/input/finetuned_wavlm/pytorch/default/1/model_finetuned_on.pth", map_location=device))  # Load directly to device


<All keys matched successfully>

### Getting mixed sounds

In [324]:
import os
import torch
import torchaudio
import librosa
import numpy as np
import mir_eval
from speechbrain.inference.separation import SepformerSeparation as separator
from tqdm import tqdm

# Paths
input_dir = '/kaggle/input/mastervox1vox2/aac/'
mixed_train_dir = '/kaggle/working/mixed_train'
mixed_test_dir = '/kaggle/working/mixed_test'
output_dir = '/kaggle/working/sepformer_output'

# Load SepFormer model
model = separator.from_hparams(source="speechbrain/sepformer-libri2mix", savedir='pretrained_models/sepformer-libri2mix', run_opts={"device":"cuda"})



In [161]:
def compute_sdr_sir_sar(reference, estimated):
    # Adjusting length of both signals
    min_length = min(len(reference), len(estimated))
    reference = reference[:min_length]
    estimated = estimated[:min_length]
    
    # mir_eval expects audio to be a 2D array (n_frames, n_channels)
    reference = np.expand_dims(reference, axis=1)
    estimated = np.expand_dims(estimated, axis=1)
    
    # Compute SDR, SIR, SAR using mir_eval
    sdr, sir, sar, _ = mir_eval.separation.bss_eval_sources(reference, estimated)
    
    return sdr[0], sir[0], sar[0]

In [260]:
import os
import numpy as np
import librosa
import torchaudio
import torch

def find_m4a_files_in_subfolders(directory):
    m4a_files = []
    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith('.m4a'):
                m4a_path = os.path.join(root, file)
                wav_path = os.path.join(vox2_wav_form, os.path.relpath(m4a_path, directory)).replace('.m4a', '.wav')
                os.makedirs(os.path.dirname(wav_path), exist_ok=True)
                torchaudio.save(wav_path, *torchaudio.load(m4a_path, format="m4a"))
                m4a_files.append(wav_path)  # Store converted WAV file path

    return m4a_files


def mix_utterances(id1, id2, mixed_train_dir):
    files_id1 = find_m4a_files_in_subfolders(id1)
    files_id2 = find_m4a_files_in_subfolders(id2)
    
    # Check if no files found
    if not files_id1 or not files_id2:
        print(f"Skipping pair {id1}, {id2} (no valid audio files)")
        return None, None, None
    
    # Randomly select files
    file1 = np.random.choice(files_id1)
    file2 = np.random.choice(files_id2)

    # Load audio using torchaudio
    audio1, sr1 = torchaudio.load(file1)
    audio2, sr2 = torchaudio.load(file2)

    # Resample to 8000Hz if needed
    resampler = torchaudio.transforms.Resample(orig_freq=sr1, new_freq=8000)
    audio1 = resampler(audio1) if sr1 != 8000 else audio1
    audio2 = resampler(audio2) if sr2 != 8000 else audio2

    # Convert to mono
    audio1 = audio1.mean(dim=0) if audio1.shape[0] > 1 else audio1.squeeze(0)
    audio2 = audio2.mean(dim=0) if audio2.shape[0] > 1 else audio2.squeeze(0)

    # Pad or truncate to the same length
    min_len = min(audio1.shape[0], audio2.shape[0])
    audio1, audio2 = audio1[:min_len], audio2[:min_len]

    # Mix the utterances and normalize
    mixed_audio = audio1 + audio2
    mixed_audio = mixed_audio / torch.max(torch.abs(mixed_audio))  # Prevents clipping

    # Ensure output directory exists
    os.makedirs(mixed_train_dir, exist_ok=True)

    # Construct the mixed file path
    mixed_file = os.path.join(mixed_train_dir, f'{os.path.basename(file1).split(".")[0]}_{os.path.basename(file2).split(".")[0]}.wav')
    # print("Mixed file:", mixed_file)

    # Convert to correct shape (1, samples) for torchaudio.save
    mixed_audio = mixed_audio.unsqueeze(0)

    # Save the mixed audio
    torchaudio.save(mixed_file, mixed_audio, 8000)
    return mixed_file, file1, file2



In [266]:
import os

# Get the list of files in mixed_test_dir
for file in os.listdir("/kaggle/working/mixed_test"):
    file_path = os.path.join("/kaggle/working/mixed_test", file)
    # Check if the file is not in mix_map, then delete it
    if file_path not in mix_map:
        os.remove(file_path)



In [261]:
import os
import random
vox2_wav_form = "/kaggle/working/vox2wavform"
mixed_test_dir = "/kaggle/working/mixed_test"
mix_map = {}  # Dictionary to store the mapping

# Get the last 50 speaker IDs (sorted)
train_ids = sorted(os.listdir(input_dir))[50:100]

# Set a random seed for reproducibility (optional)
random.seed(42)

# Generate 500 unique mixtures
num_mixtures = 500
for _ in range(num_mixtures):
    # Randomly pick two different speakers
    id1, id2 = random.sample(train_ids, 2)
    id1_path = os.path.join(input_dir, id1)
    id2_path = os.path.join(input_dir, id2)
    
    # Mix and store
    mixed_file, file1, file2 = mix_utterances(id1_path, id2_path, mixed_test_dir)
    if mixed_file:  # Store only if mixing was successful
        mix_map[mixed_file] = (file1, file2)

    print(f"Generated: {mixed_file} from {id1} and {id2}")

print(f"Total mixtures generated: {len(mix_map)}")

Generated: /kaggle/working/mixed_test/00180_00019.wav from id06484 and id03981
Generated: /kaggle/working/mixed_test/00243_00015.wav from id03789 and id07396
Generated: /kaggle/working/mixed_test/00294_00194.wav from id04478 and id04295
Generated: /kaggle/working/mixed_test/00255_00136.wav from id04276 and id04006
Generated: /kaggle/working/mixed_test/00048_00078.wav from id07396 and id03980
Generated: /kaggle/working/mixed_test/00142_00104.wav from id06816 and id07396
Generated: /kaggle/working/mixed_test/00003_00061.wav from id05816 and id03978
Generated: /kaggle/working/mixed_test/00043_00455.wav from id06104 and id05124
Generated: /kaggle/working/mixed_test/00397_00277.wav from id03839 and id03789
Generated: /kaggle/working/mixed_test/00040_00052.wav from id03978 and id04253
Generated: /kaggle/working/mixed_test/00186_00090.wav from id04276 and id05654
Generated: /kaggle/working/mixed_test/00057_00131.wav from id06209 and id03789
Generated: /kaggle/working/mixed_test/00041_00415.wa

In [165]:
# Use to clear memory
!rm -rf /kaggle/working/sepformer_output

In [267]:
import numpy as np
import torchaudio
import librosa
import os
from pesq import pesq
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Initialize variables to store accumulated metrics
total_sdr, total_sir, total_sar, total_pesq = 0, 0, 0, 0
num_files = 0

def compute_sdr_sir_sar(reference, estimated):
    sdr, sir, sar, _ = mir_eval.separation.bss_eval_sources(reference, estimated)
    return sdr[0], sir[0], sar[0]

def separate_and_evaluate(mixed_file):
    global total_sdr, total_sir, total_sar, total_pesq, num_files
    
    # Load mixed audio
    audio, sr = torchaudio.load(mixed_file)
    
    # Ensure sample rate is 8000 Hz
    resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=8000)
    audio = resampler(audio) if sr != 8000 else audio
    # Perform separation
    separated = model.separate_batch(audio.to(device))
    
    # Convert to numpy for evaluation
    separated_audio = separated.cpu().squeeze(0).numpy()

    # Extract speaker IDs from the filename
    base_name = os.path.basename(mixed_file).split(".")[0]
    speaker1_id, speaker2_id = base_name.split("_")[:2]

    # Get the original reference audio paths
    ref_audio_1_path, ref_audio_2_path = mix_map[mixed_file]
    
    if ref_audio_1_path and ref_audio_2_path:
        ref_audio_1, _ = librosa.load(ref_audio_1_path, sr=8000)
        ref_audio_2, _ = librosa.load(ref_audio_2_path, sr=8000)
        ref_audio_1 = ref_audio_1[:min(len(ref_audio_1), len(ref_audio_2))]  # Crop ref_audio_1 to the length of ref_audio_2
        ref_audio_2 = ref_audio_2[:min(len(ref_audio_1), len(ref_audio_2))]  # Crop ref_audio_2 to the length of ref_audio_1
        separated_audio_1 = separated_audio[:, 0]  # Extract the first column (speaker 1)
        separated_audio_2 = separated_audio[:, 1]  # Extract the second column (speaker 2)
        reference = np.stack((ref_audio_1, ref_audio_2), axis=0)

        # Compute SDR, SIR, SAR
        sdr, sir, sar = compute_sdr_sir_sar(reference, separated_audio.T)

        pesq_score = pesq(8000, ref_audio_1, separated_audio_1, mode='nb')  # PESQ for speaker 1
        pesq_score2 = pesq(8000, ref_audio_2, separated_audio_2, mode='nb')  # PESQ for speaker 2


        # Print the results for each file
        # print(f"Metrics for {mixed_file}:")
        # print(f"Speaker 1 - SDR: {sdr:.2f}, SIR: {sir:.2f}, SAR: {sar:.2f}, PESQ: {pesq_score:.2f}")
        
        # Accumulate the metrics
        total_sdr += sdr
        total_sir += sir
        total_sar += sar
        total_pesq += pesq_score + pesq_score2
        num_files += 1

    # Save separated audio
    sep1_path = os.path.join(output_dir, base_name + "_spk1.wav")
    sep2_path = os.path.join(output_dir, base_name + "_spk2.wav")
    
    torchaudio.save(sep1_path, torch.tensor(separated_audio[:, 0]).unsqueeze(0), 8000)
    torchaudio.save(sep2_path, torch.tensor(separated_audio[:, 1]).unsqueeze(0), 8000)

    return sep1_path, sep2_path

# Loop through all mixed files in the test set
mixed_files = [os.path.join(mixed_test_dir, file) for file in os.listdir(mixed_test_dir)]
for mixed_file in tqdm(mixed_files):
    separate_and_evaluate(mixed_file)

# Compute and print average metrics after processing all files
print("\nAverage Metrics:")
print(f"Average SDR: {total_sdr / num_files:.2f}")
print(f"Average SIR: {total_sir / num_files:.2f}")
print(f"Average SAR: {total_sar / num_files:.2f}")
print(f"Average PESQ: {total_pesq / (2 * num_files):.2f}")  # Divided by 2 because we have two speakers


100%|██████████| 499/499 [06:39<00:00,  1.25it/s]


Average Metrics:
Average SDR: 13.00
Average SIR: 22.61
Average SAR: 14.25
Average PESQ: 2.18


In [268]:
import os
import torch
import torchaudio
import numpy as np
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score

# Directories
mixed_test_dir = "/kaggle/working/mixed_test"
sepformer_output_dir = "/kaggle/working/sepformer_output"

In [292]:
# Speaker map
import os

subfolder_to_id_map = {}

for speaker_id in os.listdir("/kaggle/input/mastervox1vox2/aac"):
    speaker_path = os.path.join("/kaggle/input/mastervox1vox2/aac", speaker_id)
    if os.path.isdir(speaker_path):  
        for subfolder in os.listdir(speaker_path):
            subfolder_to_id_map[subfolder] = speaker_id


In [279]:
# Extract embedding function
import torch
import torchaudio
from transformers import WavLMModel

def enhance(input_wav):
    waveform, sample_rate = torchaudio.load(input_wav)
    enhanced_waveform = F.vad(waveform, sample_rate)
    return enhanced_waveform, sample_rate

import torchaudio
import io
from pydub import AudioSegment

def extract_speaker_embedding(model, file_path, model_type="pretrained"):
    waveform, sample_rate = torchaudio.load(file_path)  
    # Resample if needed
    resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=8000)
    waveform = resampler(waveform) if sample_rate != 8000 else waveform
    # Ensure waveform is on the correct device
    waveform = waveform.to(device) if isinstance(waveform, torch.Tensor) else torch.tensor(waveform, device=device)
    # Extract embedding
    with torch.no_grad():
        if model_type == "finetuned":
            outputs = model.base_model.base_model.model(waveform)
        else:
            outputs = model(waveform)

    return outputs.last_hidden_state.mean(dim=1).cpu().numpy()


# file_path = "/kaggle/working/vox2wavform/07pANazoyJg/00001.wav"
# file_path = "/kaggle/working/sepformer_output/00180_00019_spk1.wav"
# file_path = "/kaggle/input/mastervox1vox2/wav/id10270/5r0dWxy17C8/00006.wav"
embedding = extract_speaker_embedding(model_finetuned, file_path,"finetuned")
print(embedding.shape)


(1, 768)


In [280]:
from tqdm import tqdm

# Get unique original speakers
unique_speakers = set()
for mixed_file, (orig1, orig2) in mix_map.items():
    unique_speakers.add(orig1)
    unique_speakers.add(orig2)
unique_speakers = list(unique_speakers)

print(f"Total unique speakers: {len(unique_speakers)}")

# Compute embeddings with progress bar
embeddings_pretrained = {}
embeddings_finetuned = {}

for spk in tqdm(unique_speakers, desc="Extracting Speaker Embeddings"):
    embeddings_pretrained[spk] = extract_speaker_embedding(model_pretrained, spk,"pretrained")
    embeddings_finetuned[spk] = extract_speaker_embedding(model_finetuned, spk, "finetuned")


Total unique speakers: 947


Extracting Speaker Embeddings: 100%|██████████| 947/947 [01:19<00:00, 11.94it/s]


In [294]:
from collections import defaultdict
import numpy as np
import torch

# Initialize dictionaries to store speaker-wise embeddings
speaker_embeddings_pretrained = defaultdict(list)
speaker_embeddings_finetuned = defaultdict(list)

# Group embeddings by speaker ID
for file_path, embedding in embeddings_pretrained.items():
    speaker_id = subfolder_to_id_map[file_path.split("/")[-2]]  # Extract speaker ID from path
    speaker_embeddings_pretrained[speaker_id].append(torch.tensor(embedding))

for file_path, embedding in embeddings_finetuned.items():
    speaker_id = subfolder_to_id_map[file_path.split("/")[-2]]  # Extract speaker ID from path
    speaker_embeddings_finetuned[speaker_id].append(torch.tensor(embedding))

# Compute average embeddings for each speaker
avg_embeddings_pretrained = {spk: torch.stack(embs).mean(dim=0).numpy() 
                             for spk, embs in speaker_embeddings_pretrained.items()}

avg_embeddings_finetuned = {spk: torch.stack(embs).mean(dim=0).numpy() 
                            for spk, embs in speaker_embeddings_finetuned.items()}

print(f"Computed average embeddings for {len(avg_embeddings_pretrained)} speakers.")


Computed average embeddings for 50 speakers.


In [199]:
# Get the first key (speaker file path)
first_key = next(iter(embeddings_pretrained))

# Print the first embedding
# print(f"First Speaker: {first_key}")
# print("Embedding:", embeddings_pretrained[first_key])
# print("Embedding:", avg_embeddings_pretrained["id07414"])

In [318]:
from tqdm import tqdm
import os
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score

true_labels, pred_labels_pretrained, pred_labels_finetuned = [], [], []

for mixed_file in tqdm(os.listdir(mixed_test_dir), desc="Processing Mixed Files"):
    if not mixed_file.endswith(".wav"):
        continue
    
    orig1, orig2 = mix_map[f"/kaggle/working/mixed_test/{mixed_file}"]
    spk1_id, spk2_id = subfolder_to_id_map[orig1.split("/")[-2]], subfolder_to_id_map[orig2.split("/")[-2]]  # Extract speaker IDs

    true_labels.extend([spk1_id, spk2_id])

    for spk_num in [1, 2]:
        sep_file = os.path.join(sepformer_output_dir, f"{mixed_file[:-4]}_spk{spk_num}.wav")

        emb_sep_pretrained = extract_speaker_embedding(model_pretrained, sep_file, "pretrained")
        emb_sep_finetuned = extract_speaker_embedding(model_finetuned, sep_file, "finetuned")

        # Extract embeddings of the original audio (not averaged)
        emb_orig1_pretrained = extract_speaker_embedding(model_pretrained, orig1, "pretrained")
        emb_orig2_pretrained = extract_speaker_embedding(model_pretrained, orig2, "pretrained")

        sim_spk1 = cosine_similarity(emb_sep_pretrained, emb_orig1_pretrained)[0][0]
        sim_spk2 = cosine_similarity(emb_sep_pretrained, emb_orig2_pretrained)[0][0]

        true_label = spk1_id if sim_spk1 > sim_spk2 else spk2_id

        # Find best match using average embeddings for classification
        best_match_pretrained = min(avg_embeddings_pretrained.keys(), key=lambda spk: cosine_similarity(emb_sep_pretrained, avg_embeddings_pretrained[spk].reshape(1, -1))[0][0])
        best_match_finetuned = min(avg_embeddings_finetuned.keys(), key=lambda spk: cosine_similarity(emb_sep_finetuned, avg_embeddings_finetuned[spk].reshape(1, -1))[0][0])

        pred_labels_pretrained.append(best_match_pretrained)
        pred_labels_finetuned.append(best_match_finetuned)
        
rank1_acc_pretrained = accuracy_score(true_labels, pred_labels_pretrained)
rank1_acc_finetuned = accuracy_score(true_labels, pred_labels_finetuned)

df_acc = pd.DataFrame({
    "Model": ["Pretrained Model", "Finetuned Model"],
    "Rank-1 Accuracy": [rank1_acc_pretrained, rank1_acc_finetuned]
})

df_acc.to_csv("/kaggle/working/speaker_identification_results.csv", index=False)

print("Speaker Identification Results saved.")
print(df_acc)


Processing Mixed Files: 100%|██████████| 499/499 [02:48<00:00,  2.96it/s]

Speaker Identification Results saved.
              Model Rank-1 Accuracy
0  Pretrained Model        0.087609
1   Finetuned Model         0.24448


In [306]:
mix_map["/kaggle/working/mixed_test/00002_00108.wav"]

('/kaggle/working/vox2wavform/0Mnnngo-PkE/00002.wav',
 '/kaggle/working/vox2wavform/a85EWXgoBFQ/00108.wav')

In [316]:
from IPython.display import Audio
# merged audio
file_path = "/kaggle/working/mixed_test/00002_00108.wav"
Audio(file_path)

In [308]:
# Original 1
file_path = "/kaggle/working/vox2wavform/0Mnnngo-PkE/00002.wav"
Audio(file_path)

In [309]:
# Orginial 2
file_path = "/kaggle/working/vox2wavform/a85EWXgoBFQ/00108.wav"
Audio(file_path)


In [311]:
# Produced by sepformer 1
file_path = "/kaggle/working/sepformer_output/00002_00108_spk1.wav"
Audio(file_path)

In [312]:
# Produced by sepformer 2
file_path = "/kaggle/working/sepformer_output/00002_00108_spk2.wav"
Audio(file_path)

# Question 1 Part 4

In [319]:
import os
import torch
import torchaudio
import torchaudio.transforms as T
import numpy as np
import librosa
import soundfile as sf
from tqdm import tqdm
from pesq import pesq
from scipy.signal import resample
from sklearn.metrics.pairwise import cosine_similarity


In [320]:
def load_audio(file_path, target_sr=8000):
    waveform, sample_rate = torchaudio.load(file_path)
    if sample_rate != target_sr:
        resampler = T.Resample(orig_freq=sample_rate, new_freq=target_sr)
        waveform = resampler(waveform)
    return waveform

def extract_speaker_embedding(model, file_path, model_type="pretrained"):
    waveform = load_audio(file_path).to(device)

    with torch.no_grad():
        if model_type == "finetuned":
            outputs = model.base_model.base_model.model(waveform)
        else:
            outputs = model(waveform)

    return outputs.last_hidden_state.mean(dim=1).cpu().numpy()


In [321]:
def compute_speech_quality_metrics(reference, estimated, sample_rate=8000):
    ref, _ = librosa.load(reference, sr=sample_rate)
    est, _ = librosa.load(estimated, sr=sample_rate)

    # Ensure same length
    min_len = min(len(ref), len(est))
    ref, est = ref[:min_len], est[:min_len]

    # Compute metrics
    sir = librosa.core.snr(ref, est)
    sar = librosa.feature.rms(y=est).mean() / librosa.feature.rms(y=ref).mean()
    sdr = librosa.core.perceptual_weighting(np.abs(est - ref), np.abs(ref))
    pesq_score = pesq(sample_rate, ref, est, "wb")

    return sir, sar, sdr, pesq_score


In [322]:
def speaker_separation_and_enhancement_pipeline(mixed_test_dir, sepformer_model, speaker_model, save_path):
    results = []

    for mixed_file in tqdm(os.listdir(mixed_test_dir), desc="Processing Mixed Files"):
        if not mixed_file.endswith(".wav"):
            continue

        mixed_path = os.path.join(mixed_test_dir, mixed_file)
        
        # Step 1: Perform Speaker Separation using SepFormer
        with torch.no_grad():
            separated_speakers = sepformer_model(mixed_path)  # Assuming model returns a list of separated waveforms
        
        for i, sep_waveform in enumerate(separated_speakers):
            sep_file = os.path.join(save_path, f"{mixed_file[:-4]}_spk{i+1}.wav")
            sf.write(sep_file, sep_waveform.numpy(), 8000)

            # Step 2: Perform Speaker Identification
            emb_sep = extract_speaker_embedding(speaker_model, sep_file)
            best_match = min(avg_embeddings_pretrained.keys(), 
                             key=lambda spk: cosine_similarity(emb_sep, avg_embeddings_pretrained[spk].reshape(1, -1))[0][0])

            # Step 3: Perform Speech Enhancement using SepFormer (again)
            enhanced_waveform = sepformer_model(sep_file)  # Assuming the model can enhance speech too
            enhanced_file = os.path.join(save_path, f"{mixed_file[:-4]}_spk{i+1}_enhanced.wav")
            sf.write(enhanced_file, enhanced_waveform.numpy(), 8000)

            # Step 4: Evaluate Enhancement Performance
            sir, sar, sdr, pesq_score = compute_speech_quality_metrics(sep_file, enhanced_file)

            results.append({
                "Mixed File": mixed_file,
                "Speaker ID": best_match,
                "SIR": sir,
                "SAR": sar,
                "SDR": sdr,
                "PESQ": pesq_score
            })

    return results
